# tinyforge on Colab: Linux + real GPU smoke test

Runtime -> Change runtime type -> **T4 GPU**, then Run all. Takes about 15-25 minutes. Each section prints PASS/FAIL
at the end; copy the final summary cell's output back to the maintainer. One GPU only: multi-GPU and Kubernetes
are not covered here.

In [ ]:
import subprocess, json, time
RESULTS = {}
def sh(cmd, name=None, timeout=3600):
    t = time.time()
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    out = (p.stdout + p.stderr)[-3000:]
    print(f"$ {cmd}\n{out}\n[exit {p.returncode}, {time.time()-t:.0f}s]")
    if name: RESULTS[name] = p.returncode == 0
    return p.returncode
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv; python --version; uname -a")

In [ ]:
sh('pip install -q "tinyforge[finetune]"', "install")
sh("pip show tinyforge | head -3; tinyforge --version", "version")

In [ ]:
sh("tinyforge doctor", "doctor")
sh("tinyforge memory", "memory")

In [ ]:
# From-scratch path: doctor -> data -> plan -> train -> eval (eval exits non-zero if a quality gate fails)
sh("tinyforge pipeline --preset micro --steps 300", "pipeline_scratch")

In [ ]:
# Fine-tune a pretrained model (LoRA): downloads SmolLM2-360M-Instruct
sh("tinyforge ft pipeline --steps 100", "ft_pipeline")
sh('tinyforge ft generate "Explain hash tables in two sentences"', "ft_generate")

In [ ]:
# Serve + OpenAI-compatible API (auth off: local, loopback only)
import os, subprocess, time, urllib.request
env = dict(os.environ, TINYFORGE_AUTH="off")
srv = subprocess.Popen(["tinyforge", "serve", "--port", "8000"], env=env, stdout=open("serve.log", "w"), stderr=subprocess.STDOUT)
ok = False
for _ in range(90):
    try:
        ok = urllib.request.urlopen("http://127.0.0.1:8000/healthz", timeout=2).status == 200
        if ok: break
    except Exception:
        time.sleep(2)
print("healthz:", ok)
RESULTS["serve_health"] = ok
if ok:
    for p in ("/v1/models", "/metrics"):
        try:
            r = urllib.request.urlopen("http://127.0.0.1:8000" + p, timeout=10)
            print(p, r.status, r.read()[:300])
            RESULTS["serve" + p] = r.status == 200
        except Exception as e:
            print(p, "ERR", e); RESULTS["serve" + p] = False
srv.terminate()
print(open("serve.log").read()[-1500:])

In [ ]:
sh("pip install -q pytest pytest-cov && pip download tinyforge==0.1.0 --no-deps --no-binary :all: -d src_dl -q && cd src_dl && tar xzf tinyforge-0.1.0.tar.gz && cd tinyforge-0.1.0 && pip install -q -e . && python -m pytest -q -x -m 'not gpu' tests 2>&1 | tail -15", "test_suite_sdist", timeout=3000)

In [ ]:
print(json.dumps(RESULTS, indent=2))
print("ALL PASS" if all(RESULTS.values()) else "SOME FAILED: " + ", ".join(k for k, v in RESULTS.items() if not v))